# Smart Hygiene Risk Prediction System
## Exploratory Data Analysis Notebook

**Pugarch Internship — Day 10 AI/ML Assignment**

> ⚠️ **DISCLAIMER**: The dataset used in this notebook is **entirely synthetic** and generated programmatically for educational purposes. The target variable `hygiene_risk` is **rule-generated**, not independently verified by any public-health authority. No real facilities, locations, or inspection outcomes are represented.

---

### Contents
1. Dataset loading and shape
2. Summary statistics
3. Missing-value analysis
4. Duplicate-record analysis
5. Target distribution
6. Cleanliness score distribution
7. Odor score distribution
8. Waste level analysis
9. Complaint distribution
10. Feature vs. target relationships
11. Correlation heatmap
12. Key findings

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 100, 'savefig.bbox': 'tight'})

PALETTE = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
DATA_PATH = os.path.join('..', 'data', 'hygiene_inspections.csv')
OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports OK')

## 1. Dataset Loading and Shape

In [ ]:
# ── Cell 2: Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 2. Summary Statistics

In [ ]:
# ── Cell 3: Summary stats ─────────────────────────────────────────────────────
print('--- Dtypes ---')
print(df.dtypes)
print('\n--- Numeric summary ---')
df.describe(include=[np.number]).T

## 3. Missing-Value Analysis

In [ ]:
# ── Cell 4: Missing values ────────────────────────────────────────────────────
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing': missing, 'Pct %': pct})
print('Missing values per column:')
display(miss_df[miss_df['Missing'] > 0] if miss_df['Missing'].any() else 'None detected')

# Visualise
fig, ax = plt.subplots(figsize=(8, 3))
miss_df[miss_df['Missing'] > 0]['Pct %'].plot(kind='barh', ax=ax, color='#e67e22')
ax.set_title('Missing Value Percentage by Column')
ax.set_xlabel('Missing (%)')
plt.tight_layout()
plt.show()

## 4. Duplicate-Record Analysis

In [ ]:
# ── Cell 5: Duplicates ────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f'Removed duplicates. New shape: {df.shape}')

## 5. Target Distribution — Hygiene Risk

In [ ]:
# ── Cell 6: Risk distribution ─────────────────────────────────────────────────
order = ['Low', 'Medium', 'High']
counts = df['hygiene_risk'].value_counts().reindex(order)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(order, counts, color=[PALETTE[c] for c in order], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontweight='bold')
ax.set_title('Hygiene Risk Category Distribution\n(Rule-Generated Labels)', pad=12)
ax.set_xlabel('Hygiene Risk')
ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.15)
sns.despine()
fig.savefig(os.path.join(OUTPUT_DIR, 'risk_distribution.png'))
plt.show()
print(counts)

## 6. Cleanliness Score Distribution

In [ ]:
# ── Cell 7: Cleanliness distribution ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for cat in ['Low', 'Medium', 'High']:
    subset = df[df['hygiene_risk'] == cat]['cleanliness_score'].dropna()
    ax.hist(subset, bins=20, alpha=0.55, color=PALETTE[cat], label=cat, edgecolor='white')
ax.set_title('Cleanliness Score Distribution by Hygiene Risk', pad=12)
ax.set_xlabel('Cleanliness Score (0–100)')
ax.set_ylabel('Frequency')
ax.legend(title='Hygiene Risk')
sns.despine()
fig.savefig(os.path.join(OUTPUT_DIR, 'cleanliness_distribution.png'))
plt.show()

## 7. Odor Score Distribution

In [ ]:
# ── Cell 8: Odor score box plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
data = [df[df['hygiene_risk'] == c]['odor_score'].dropna() for c in ['Low', 'Medium', 'High']]
bp = ax.boxplot(data, labels=['Low', 'Medium', 'High'], patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, cat in zip(bp['boxes'], ['Low', 'Medium', 'High']):
    patch.set_facecolor(PALETTE[cat])
    patch.set_alpha(0.75)
ax.set_title('Odor Score Distribution by Hygiene Risk', pad=12)
ax.set_xlabel('Hygiene Risk Category')
ax.set_ylabel('Odor Score (0–10)')
sns.despine()
plt.show()

## 8. Waste Level Analysis

In [ ]:
# ── Cell 9: Waste level ───────────────────────────────────────────────────────
cross = pd.crosstab(df['waste_level'], df['hygiene_risk'])
cross = cross.reindex(['Low', 'Medium', 'High'], axis=1).reindex(['Low', 'Medium', 'High'], axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
cross.plot(kind='bar', ax=ax, color=[PALETTE[c] for c in cross.columns],
           edgecolor='white', width=0.6)
ax.set_title('Waste Level vs. Hygiene Risk Category', pad=12)
ax.set_xlabel('Waste Level')
ax.set_ylabel('Count')
ax.legend(title='Hygiene Risk', bbox_to_anchor=(1.01, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
sns.despine()
plt.show()

## 9. Complaint Distribution

In [ ]:
# ── Cell 10: Complaints ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for cat in ['Low', 'Medium', 'High']:
    subset = df[df['hygiene_risk'] == cat]['complaints'].dropna()
    ax.hist(subset, bins=20, alpha=0.55, color=PALETTE[cat], label=cat, edgecolor='white')
ax.set_title('Complaint Distribution by Hygiene Risk', pad=12)
ax.set_xlabel('Number of Complaints')
ax.set_ylabel('Frequency')
ax.legend(title='Hygiene Risk')
sns.despine()
plt.show()

## 10. Feature vs. Target Relationships

In [ ]:
# ── Cell 11: Violin plots ─────────────────────────────────────────────────────
features_to_plot = ['cleanliness_score', 'odor_score', 'footfall', 'complaints']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

order = ['Low', 'Medium', 'High']
for i, feat in enumerate(features_to_plot):
    sns.violinplot(
        data=df, x='hygiene_risk', y=feat,
        order=order, palette=PALETTE,
        ax=axes[i], inner='box'
    )
    axes[i].set_title(f'{feat} vs. Hygiene Risk')
    axes[i].set_xlabel('Hygiene Risk')
    axes[i].set_ylabel(feat)

plt.suptitle('Feature vs. Hygiene Risk Relationships', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. Correlation Heatmap

In [ ]:
# ── Cell 12: Correlation heatmap ──────────────────────────────────────────────
numeric_cols = ['cleanliness_score', 'odor_score', 'footfall', 'complaints']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr, ax=ax, annot=True, fmt='.2f',
    cmap='RdYlGn_r', center=0,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap — Numeric Features', pad=12)
fig.savefig(os.path.join(OUTPUT_DIR, 'correlation_heatmap.png'))
plt.show()

## 12. Key Findings

| Observation | Interpretation |
|-------------|----------------|
| Higher cleanliness scores → Lower risk | Reflects the labeling rule (negative coefficient) |
| Higher odor scores → Higher risk | Reflects the labeling rule (positive coefficient) |
| "High" waste level → More High-risk records | Waste level contributes to the risk score |
| Water unavailability → Higher risk | Water availability adds a penalty in the rule |
| Numeric features have low inter-correlation | Each feature contributes independently to risk score |

> **Important caveat**: These observations reflect the structure of the synthetic labeling rule, not independently observed real-world patterns. Correlations do not imply causation, and these findings do not apply to actual hygiene inspections.